# Лабораторная работа №3
## Сравнение реализаций нейронных сетей: Keras vs PyTorch vs собственная реализация

**Выполнил:** Привалихин Дмитрий Сергеевич, ИСИб-23-1

**Вариант:** tanh, α=3, T0=0

**Цель работы:**
- Реализовать нейронную сеть для классификации символов тремя способами
- Сравнить эффективность реализаций на Keras, PyTorch и собственной реализации
- Исследовать влияние функции активации tanh с коэффициентом α=3 на обучение сети

---
## 1. Конфигурация и импорт библиотек

In [ ]:
import os
import random
import numpy as np
import copy
from PIL import Image, ImageDraw, ImageFont, ImageOps, ImageFilter

# Размер изображения символа (32x32 пикселя)
IMG_SIZE = 32
# Алфавит из 11 букв для классификации
ALPHABET = "йклмнопрсту"
# Количество примеров для каждой буквы
NUM_SAMPLES_PER_LETTER = 1000
# Директория для сохранения изображений
DATASET_DIR = "dataset_images"

# Параметры функции активации tanh(α*(x-T0))
ALPHA = 3
T0 = 0

### Пояснения к конфигурации:
- **IMG_SIZE=32** — размер входных изображений, достаточный для распознавания букв
- **ALPHABET** — набор из 11 кириллических букв, похожих по написанию
- **NUM_SAMPLES_PER_LETTER=1000** — по 1000 примеров на класс для качественного обучения
- **ALPHA=3** — коэффициент усиления для функции tanh, увеличивает крутизну функции активации

---
## 2. Генерация датасета

In [ ]:
def get_local_fonts():
    """Поиск доступных шрифтов .ttf в проекте и системной папке."""
    font_paths = [f for f in os.listdir('.') if f.endswith('.ttf')]
    if not font_paths:
        sys_path = "C:\\Windows\\Fonts\\"
        common_fonts = ['arial.ttf', 'times.ttf', 'verdana.ttf']
        font_paths = [os.path.join(sys_path, f) for f in common_fonts if os.path.exists(os.path.join(sys_path, f))]
    if not font_paths:
        raise RuntimeError("Файлы .ttf не найдены!")
    return font_paths

def generate_char_image(char, font_path, save_path=None):
    """Генерация одного изображения символа с аугментацией."""
    canvas_size = IMG_SIZE * 3
    bg_color = random.randint(235, 255)
    img = Image.new('L', (canvas_size, canvas_size), bg_color)
    draw = ImageDraw.Draw(img)

    # Случайный размер шрифта с вариативностью ±30%
    font_size = random.randint(int(IMG_SIZE * 0.9), int(IMG_SIZE * 1.3))
    font = ImageFont.truetype(font_path, font_size)

    # Центрирование символа
    bbox = draw.textbbox((0, 0), char, font=font)
    w, h = bbox[2] - bbox[0], bbox[3] - bbox[1]
    x = (canvas_size - w) / 2 - bbox[0]
    y = (canvas_size - h) / 2 - bbox[1]
    draw.text((x, y), char, font=font, fill=0)

    # Поворот на случайный угол до ±15°
    img = img.rotate(random.uniform(-15, 15), resample=Image.BICUBIC, fillcolor=bg_color)
    # Размытие с вероятностью 30%
    if random.random() < 0.3:
        img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, 0.8)))

    # Кадрирование центра с небольшим смещением
    center = canvas_size // 2
    s = IMG_SIZE // 2
    jx, jy = random.randint(-2, 2), random.randint(-2, 2)
    img = img.crop((center - s + jx, center - s + jy, center + s + jx, center + s + jy))

    if save_path:
        img.save(save_path)

    # Инверсия и нормализация к [0, 1]
    img = ImageOps.invert(img)
    arr = np.array(img).astype(np.float32) / 255.0
    # Добавление шума с вероятностью 40%
    if random.random() < 0.4:
        arr += np.random.normal(0, 0.01, arr.shape)

    # Бинаризация по порогу 0.4
    arr = (np.clip(arr, 0, 1) > 0.4).astype(np.float32)
    return arr.flatten()

def generate_dataset():
    """Генерация полного датасета для всех букв."""
    fonts = get_local_fonts()
    all_images = []
    all_labels = []

    if not os.path.exists(DATASET_DIR):
        os.makedirs(DATASET_DIR)
        print(f"Создана директория: {DATASET_DIR}")

    print("Генерация данных...")
    for label, char in enumerate(ALPHABET):
        char_dir = os.path.join(DATASET_DIR, char)
        if not os.path.exists(char_dir):
            os.makedirs(char_dir)

        for i in range(NUM_SAMPLES_PER_LETTER):
            try:
                f = random.choice(fonts)
                c = char.upper() if random.random() < 0.3 else char
                img_name = f"img_{i}.png"
                save_path = os.path.join(char_dir, img_name)
                img_vector = generate_char_image(c, f, save_path=save_path)
                all_images.append(img_vector)
                all_labels.append(label)
            except Exception as e:
                continue

    print(f"Генерация завершена. Всего сохранено {len(all_images)} изображений.")
    return np.array(all_images), np.array(all_labels)

# Запуск генерации
X, y = generate_dataset()

### Пояснения к генерации данных:
1. **Аугментация данных** — применяется для увеличения разнообразия обучающей выборки:
   - Случайные шрифты (arial, times, verdana)
   - Поворот изображения на ±15°
   - Размытие Гаусса (30% вероятность)
   - Добавление гауссовского шума (40% вероятность)
   - Случайное смещение при кадрировании

2. **Предобработка**:
   - Инверсия цветов (чёрный текст на белом фоне → белый на чёрном)
   - Нормализация к диапазону [0, 1]
   - Бинаризация по порогу 0.4

3. **Результат**: 11000 изображений (11 букв × 1000 примеров)

---
## 3. Собственная реализация нейронной сети

In [ ]:
class Network:
    """Полносвязная нейронная сеть с функцией активации tanh(α*(x-T0))."""
    
    def __init__(self, sizes):
        """Инициализация слоёв сети.
        
        Args:
            sizes: список размеров слоёв [input, hidden1, hidden2, ..., output]
        """
        self.num_layers = len(sizes)
        self.sizes = sizes
        # Инициализация смещений случайными значениями
        self.biases = [np.random.randn(y, 1) for y in sizes[1:]]
        # Инициализация весов методом He initialization
        self.weights = [np.random.randn(y, x) * np.sqrt(2 / x) for x, y in zip(sizes[:-1], sizes[1:])]
        self.history = []

    def tanh(self, z):
        """Функция активации tanh с коэффициентом α=3."""
        return np.tanh(ALPHA * (z - T0))

    def tanh_prime(self, z):
        """Производная функции активации tanh."""
        tanh_val = self.tanh(z)
        return ALPHA * (1 - tanh_val ** 2)

    def softmax(self, z):
        """Функция softmax для выходного слоя (классификация)."""
        exp_z = np.exp(z - np.max(z, axis=0, keepdims=True))
        return exp_z / np.sum(exp_z, axis=0, keepdims=True)

    def feedforward(self, a):
        """Прямое распространение сигнала через сеть."""
        for i, (b, w) in enumerate(zip(self.biases, self.weights)):
            z = np.dot(w, a) + b
            # softmax на последнем слое, tanh на остальных
            a = self.softmax(z) if i == self.num_layers - 2 else self.tanh(z)
        return a

    def SGD(self, training_data, epochs, mini_batch_size, eta,
            test_data=None, early_stopping=True, patience=5, monitor='val_loss'):
        """Обучение сети методом стохастического градиентного спуска.
        
        Args:
            training_data: обучающая выборка (X, y)
            epochs: количество эпох обучения
            mini_batch_size: размер мини-батча
            eta: скорость обучения (learning rate)
            test_data: проверочная выборка
            early_stopping: флаг ранней остановки
            patience: количество эпох ожидания улучшения
            monitor: метрика для отслеживания ('val_loss' или 'val_acc')
        """
        n = len(training_data)
        if test_data:
            n_test = len(test_data)

        best_weights = copy.deepcopy(self.weights)
        best_biases = copy.deepcopy(self.biases)
        best_metric = float('inf') if monitor == 'val_loss' else 0
        wait = 0

        for epoch in range(epochs):
            random.shuffle(training_data)
            mini_batches = [training_data[k:k + mini_batch_size] for k in range(0, n, mini_batch_size)]

            for mini_batch in mini_batches:
                x_batch = np.hstack([x for x, y in mini_batch])
                y_batch = np.hstack([y for x, y in mini_batch])
                self.update_mini_batch(x_batch, y_batch, eta)

            train_acc = self.evaluate([(x, np.argmax(y)) for x, y in training_data]) / n
            train_loss = self.compute_loss(training_data)

            val_acc = val_loss = None
            if test_data:
                val_acc = self.evaluate(test_data) / n_test
                val_loss = self.compute_loss(test_data, is_test=True)

            log = {
                "epoch": epoch + 1,
                "train_loss": train_loss,
                "train_acc": train_acc,
                "val_loss": val_loss,
                "val_acc": val_acc
            }
            self.history.append(log)

            print_str = f"Epoch {epoch+1}/{epochs} - loss: {train_loss:.4f} - acc: {train_acc:.4f}"
            if test_data:
                print_str += f" - val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}"
            print(print_str)

            if early_stopping and test_data:
                current_metric = val_loss if monitor == 'val_loss' else val_acc
                is_improved = (current_metric < best_metric) if monitor == 'val_loss' else (current_metric > best_metric)

                if is_improved:
                    best_metric = current_metric
                    best_weights = copy.deepcopy(self.weights)
                    best_biases = copy.deepcopy(self.biases)
                    wait = 0
                else:
                    wait += 1
                    if wait >= patience:
                        print(f"Early stopping at epoch {epoch+1}. Best {monitor}: {best_metric:.4f}")
                        break

        self.weights = best_weights
        self.biases = best_biases

    def update_mini_batch(self, x_batch, y_batch, eta):
        """Обновление весов на основе мини-батча."""
        nabla_b, nabla_w = self.backprop(x_batch, y_batch)
        batch_size = x_batch.shape[1]
        self.weights = [w - (eta / batch_size) * nw for w, nw in zip(self.weights, nabla_w)]
        self.biases = [b - (eta / batch_size) * nb for b, nb in zip(self.biases, nabla_b)]

    def backprop(self, x_batch, y_batch):
        """Алгоритм обратного распространения ошибки."""
        nabla_b = [np.zeros(b.shape) for b in self.biases]
        nabla_w = [np.zeros(w.shape) for w in self.weights]

        activation = x_batch
        activations = [x_batch]
        zs = []

        # Прямой проход
        for i, (b, w) in enumerate(zip(self.biases, self.weights)):
            z = np.dot(w, activation) + b
            zs.append(z)
            activation = self.softmax(z) if i == self.num_layers - 2 else self.tanh(z)
            activations.append(activation)

        # Обратный проход (вычисление градиентов)
        delta = activations[-1] - y_batch  # Градиент для выходного слоя
        nabla_b[-1] = np.sum(delta, axis=1, keepdims=True)
        nabla_w[-1] = np.dot(delta, activations[-2].T)

        for l in range(2, self.num_layers):
            z = zs[-l]
            sp = self.tanh_prime(z)
            delta = np.dot(self.weights[-l + 1].T, delta) * sp
            nabla_b[-l] = np.sum(delta, axis=1, keepdims=True)
            nabla_w[-l] = np.dot(delta, activations[-l - 1].T)

        return nabla_b, nabla_w

    def evaluate(self, test_data):
        """Оценка точности на тестовых данных."""
        test_results = [(np.argmax(self.feedforward(x)), y) for (x, y) in test_data]
        return sum(int(x == y) for (x, y) in test_results)

    def compute_loss(self, data, is_test=False):
        """Вычисление функции потерь (кросс-энтропия)."""
        total_loss = 0.0
        for x, y in data:
            if is_test:
                y = self.vectorize_label(y)
            output = self.feedforward(x)
            loss = -np.sum(y * np.log(output + 1e-8))
            total_loss += loss
        return total_loss / len(data)

    def vectorize_label(self, label):
        """Преобразование метки в one-hot вектор."""
        e = np.zeros((self.sizes[-1], 1))
        e[label] = 1.0
        return e

### Пояснения к собственной реализации:

1. **Архитектура сети**:
   - Входной слой: 1024 нейрона (32×32 пикселя)
   - Скрытые слои: 128 и 64 нейрона
   - Выходной слой: 11 нейронов (по числу классов)

2. **Функция активации**:
   - `tanh(3x)` — модифицированная гиперболическая функция
   - Коэффициент α=3 увеличивает крутизну, ускоряя обучение
   - Производная: `3 × (1 - tanh²(3x))`

3. **Алгоритм обучения**:
   - SGD (стохастический градиентный спуск) с мини-батчами
   - Backpropagation для вычисления градиентов
   - Early stopping для предотвращения переобучения
   - Инициализация весов по He (для tanh)

4. **Функция потерь**: кросс-энтропия для многоклассовой классификации

---
## 4. Подготовка данных для обучения

In [ ]:
# Разбиение на обучающую и тестовую выборки (80/20)
num_classes = len(ALPHABET)
test_size = 0.2
n_samples = len(X)
indices = np.random.permutation(n_samples)
test_count = int(test_size * n_samples)

test_indices = indices[:test_count]
train_indices = indices[test_count:]

X_train, X_test = X[train_indices], X[test_indices]
y_train, y_test = y[train_indices], y[test_indices]

# One-hot кодирование меток для обучения
def to_categorical(labels, num_classes):
    result = np.zeros((len(labels), num_classes))
    for i, label in enumerate(labels):
        result[i, label] = 1.0
    return result

y_train_cat = to_categorical(y_train, num_classes)

# Преобразование данных для собственной сети
X_train = [x.reshape(-1, 1) for x in X_train]
X_test = [x.reshape(-1, 1) for x in X_test]
y_train = [y.reshape(-1, 1) for y in y_train_cat]
training_data = list(zip(X_train, y_train))
test_data = list(zip(X_test, y_test))

input_size = IMG_SIZE * IMG_SIZE

print(f"Обучающая выборка: {len(training_data)} примеров")
print(f"Тестовая выборка: {len(test_data)} примеров")

### Пояснения к подготовке данных:
- **Разбиение 80/20**: 8800 примеров для обучения, 2200 для тестирования
- **One-hot кодирование**: метки преобразуются в векторы [0,0,1,0,...] для обучения
- **Формат данных**: для собственной сети — столбцы (1024×1), для Keras/PyTorch — плоские массивы

---
## 5. Реализация на Keras (TensorFlow)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Кастомная функция активации: tanh(3x)
def custom_tanh(x):
    return tf.nn.tanh(3.0 * x)

# Регистрация функции для сохранения модели
keras.utils.get_custom_objects()['custom_tanh'] = custom_tanh

# Подготовка данных для Keras (формат: 32x32x1)
X_train_k = np.array([x.reshape(IMG_SIZE, IMG_SIZE, 1) for x in X_train]).astype('float32')
X_test_k = np.array([x.reshape(IMG_SIZE, IMG_SIZE, 1) for x in X_test]).astype('float32')

# Метки в виде индексов (0, 1, 2, ..., 10)
y_train_idx = np.array([np.argmax(y) for y in y_train_cat]).astype('int64')
y_test_idx = y_test.astype('int64')

# Архитектура модели
keras_model = keras.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1)),
    layers.Flatten(),
    
    # Скрытый слой 1: 128 нейронов + custom_tanh
    layers.Dense(128, kernel_initializer='glorot_uniform'),
    layers.Activation(custom_tanh),
    
    # Скрытый слой 2: 64 нейрона + custom_tanh
    layers.Dense(64, kernel_initializer='glorot_uniform'),
    layers.Activation(custom_tanh),
    
    # Выходной слой: softmax для классификации
    layers.Dense(num_classes, activation='softmax')
])

# Компиляция модели
keras_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0005),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\n=== Архитектура модели Keras ===")
keras_model.summary()

### Пояснения к реализации на Keras:

1. **Архитектура**:
   - Input: 32×32×1 (изображения в оттенках серого)
   - Flatten: преобразование в вектор 1024
   - Dense(128) + Activation(custom_tanh): первый скрытый слой
   - Dense(64) + Activation(custom_tanh): второй скрытый слой
   - Dense(11) + softmax: выходной слой для классификации

2. **Функция активации**:
   - `custom_tanh(x) = tanh(3x)` — кастомная функция
   - Регистрируется через `get_custom_objects()` для сохранения

3. **Компиляция**:
   - Optimizer: Adam с lr=0.0005 (меньше из-за α=3)
   - Loss: sparse_categorical_crossentropy (для целочисленных меток)
   - Metrics: accuracy

In [ ]:
# Обучение модели Keras
print("\n=== ОБУЧЕНИЕ KERAS (tanh, alpha=3) ===")
keras_history = keras_model.fit(
    X_train_k, y_train_idx,
    epochs=150,
    batch_size=32,
    validation_data=(X_test_k, y_test_idx),
    shuffle=True,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=30,
            restore_best_weights=True
        )
    ],
    verbose=1
)

# Результаты обучения
print(f"\n=== Результаты Keras ===")
print(f"Лучшая точность на обучении: {max(keras_history.history['accuracy']):.4f}")
print(f"Лучшая точность на валидации: {max(keras_history.history['val_accuracy']):.4f}")

### Пояснения к обучению Keras:
- **epochs=150**: максимальное количество эпох
- **batch_size=32**: размер мини-батча
- **EarlyStopping**: остановка если val_loss не улучшается 30 эпох
- **restore_best_weights=True**: восстановление весов лучшей модели

---
## 6. Реализация на PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Проверка доступности GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используемое устройство: {device}")

class PyTorchNet(nn.Module):
    """Нейронная сеть на PyTorch с активацией tanh(3x)."""
    
    def __init__(self, input_size, num_classes):
        super(PyTorchNet, self).__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, num_classes)
        # Инициализация весов Xavier
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.xavier_uniform_(self.fc2.weight)
        nn.init.xavier_uniform_(self.fc3.weight)

    def forward(self, x):
        """Прямой проход через сеть."""
        # custom_tanh(x) = tanh(3.0 * x)
        x = torch.tanh(3.0 * self.fc1(x))
        x = torch.tanh(3.0 * self.fc2(x))
        x = self.fc3(x)  # Без активации (CrossEntropyLoss включает softmax)
        return x

# Подготовка данных для PyTorch
X_train_arr = np.array([x.flatten() for x in X_train])
X_test_arr = np.array([x.flatten() for x in X_test])

# Метки в виде индексов
y_train_indices = np.array([np.argmax(y) for y in y_train_cat], dtype=np.int64)
y_test_indices = y_test.astype(np.int64)

# Преобразование в тензоры
X_train_pt = torch.FloatTensor(X_train_arr).to(device)
y_train_pt = torch.LongTensor(y_train_indices).to(device)
X_test_pt = torch.FloatTensor(X_test_arr).to(device)
y_test_pt = torch.LongTensor(y_test_indices).to(device)

# Создание DataLoader для мини-батчей
train_dataset = TensorDataset(X_train_pt, y_train_pt)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Создание модели
pt_model = PyTorchNet(input_size, num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(pt_model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=7, min_lr=0.00001
)

print("\n=== Архитектура модели PyTorch ===")
print(pt_model)

### Пояснения к реализации на PyTorch:

1. **Архитектура**:
   - fc1: Linear(1024 → 128) + tanh(3x)
   - fc2: Linear(128 → 64) + tanh(3x)
   - fc3: Linear(64 → 11) — без активации (включена в loss)

2. **Функция активации**:
   - `torch.tanh(3.0 * x)` — встроено в forward()

3. **Подготовка данных**:
   - TensorDataset и DataLoader для эффективной загрузки батчей
   - LongTensor для меток (требуется CrossEntropyLoss)

4. **Функция потерь**: CrossEntropyLoss (включает softmax + NLLLoss)

In [ ]:
# Обучение модели PyTorch
print("\n=== ОБУЧЕНИЕ PYTORCH (tanh, alpha=3) ===")
best_val_loss = float('inf')
best_model_state = None
patience = 30
wait = 0

for epoch in range(150):
    pt_model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = pt_model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()
    
    # Валидация
    pt_model.eval()
    with torch.no_grad():
        val_outputs = pt_model(X_test_pt)
        val_loss = criterion(val_outputs, y_test_pt).item()
        _, val_predicted = torch.max(val_outputs.data, 1)
        val_acc = (val_predicted == y_test_pt).sum().item() / len(y_test_pt)
    
    scheduler.step(val_loss)
    
    train_acc = correct / total
    print(f"Epoch {epoch+1}/150 - loss: {total_loss/len(train_loader):.4f} - acc: {train_acc:.4f} - val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}")
    
    # Early stopping с сохранением лучшей модели
    if val_loss < best_val_loss - 0.0001:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(pt_model.state_dict())
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping at epoch {epoch+1}. Best val_loss: {best_val_loss:.4f}")
            break

# Восстановление лучшей модели
if best_model_state is not None:
    pt_model.load_state_dict(best_model_state)

print(f"\n=== Результаты PyTorch ===")
print(f"Лучший val_loss: {best_val_loss:.4f}")

### Пояснения к обучению PyTorch:
- **Ручной цикл обучения**: полный контроль над процессом
   - `optimizer.zero_grad()`: обнуление градиентов
   - `loss.backward()`: вычисление градиентов
   - `optimizer.step()`: обновление весов
- **Scheduler**: уменьшение lr в 2 раза если val_loss не улучшается 7 эпох
- **Early stopping**: сохранение лучшей модели и восстановление после обучения

---
## 7. Сохранение моделей

In [ ]:
import pickle

# Сохранение модели Keras
keras_model.save('model_keras.h5')
print("Keras модель сохранена в model_keras.h5")

# Сохранение модели PyTorch
torch.save(pt_model.state_dict(), 'model_pytorch.pth')
print("PyTorch модель сохранена в model_pytorch.pth")

print("\n=== Модели сохранены ===")

### Пояснения к сохранению:
- **Keras**: HDF5 формат с архитектурой, весами и историей
- **PyTorch**: state_dict() — только веса модели

---
## 8. Приложение для тестирования моделей (рисовалка)

In [ ]:
import tkinter as tk
from tkinter import ttk
from PIL import Image, ImageDraw, ImageTk, ImageOps

# Предсказатель для Keras
class KerasPredictor:
    def __init__(self, model, alphabet):
        self.model = model
        self.alphabet = alphabet
    
    def predict_all(self, processed_img):
        img = processed_img.reshape(1, IMG_SIZE, IMG_SIZE, 1)
        output = self.model.predict(img, verbose=0)[0]
        results = [(self.alphabet[i], float(prob)) for i, prob in enumerate(output)]
        return sorted(results, key=lambda x: x[1], reverse=True)

# Предсказатель для PyTorch
class PyTorchPredictor:
    def __init__(self, model, alphabet, device):
        self.model = model
        self.alphabet = alphabet
        self.device = device
        self.model.eval()
    
    def predict_all(self, processed_img):
        img = torch.FloatTensor(processed_img.flatten()).unsqueeze(0).to(self.device)
        with torch.no_grad():
            output = torch.softmax(self.model(img), dim=1)[0].cpu().numpy()
        results = [(self.alphabet[i], float(prob)) for i, prob in enumerate(output)]
        return sorted(results, key=lambda x: x[1], reverse=True)

# Графическое приложение
class DrawingApp:
    def __init__(self, predictor, alphabet, title):
        self.predictor = predictor
        self.alphabet = alphabet
        self.img_size = 32
        self.canvas_size = 400
        self.brush_size = 18
        
        self.root = tk.Tk()
        self.root.title(title)
        self.root.configure(bg='#f0f0f0')
        
        main_frame = ttk.Frame(self.root, padding="10")
        main_frame.grid(row=0, column=0)

        self.canvas = tk.Canvas(main_frame, width=self.canvas_size, height=self.canvas_size,
                               bg='white', cursor='pencil', highlightthickness=2)
        self.canvas.grid(row=0, column=0, rowspan=10, padx=10)
        
        results_frame = ttk.LabelFrame(main_frame, text=" Вероятности ", padding="10")
        results_frame.grid(row=0, column=1, rowspan=10, sticky='ns', padx=10)
        
        self.prob_labels = {}
        self.prob_bars = {}
        
        for i, char in enumerate(self.alphabet):
            lbl = ttk.Label(results_frame, text=f"'{char}':", font=('Courier', 12))
            lbl.grid(row=i, column=0, sticky='e', pady=2)
            bar = ttk.Progressbar(results_frame, orient='horizontal', length=150, mode='determinate')
            bar.grid(row=i, column=1, padx=5)
            percent_lbl = ttk.Label(results_frame, text="0.0%", width=8)
            percent_lbl.grid(row=i, column=2, sticky='w')
            self.prob_labels[char] = percent_lbl
            self.prob_bars[char] = bar

        controls = ttk.Frame(main_frame)
        controls.grid(row=11, column=0, columnspan=2, pady=10, sticky='ew')
        ttk.Button(controls, text='Очистить', command=self.clear_canvas).pack(side='left', padx=5)
        
        self.preview_canvas = tk.Canvas(controls, width=64, height=64, bg='black')
        self.preview_canvas.pack(side='right', padx=20)
        ttk.Label(controls, text="Вход ИИ:").pack(side='right')

        self.image = Image.new('L', (self.canvas_size, self.canvas_size), 255)
        self.draw = ImageDraw.Draw(self.image)
        
        self.canvas.bind('<B1-Motion>', self.paint)
        self.canvas.bind('<ButtonRelease-1>', lambda e: self.process_and_predict())
        self.last_x, self.last_y = None, None

    def paint(self, event):
        if self.last_x and self.last_y:
            self.canvas.create_line(self.last_x, self.last_y, event.x, event.y,
                                   width=self.brush_size, fill='black', capstyle=tk.ROUND)
            self.draw.line([self.last_x, self.last_y, event.x, event.y],
                          fill=0, width=self.brush_size)
        self.last_x, self.last_y = event.x, event.y

    def clear_canvas(self):
        self.canvas.delete("all")
        self.image = Image.new('L', (self.canvas_size, self.canvas_size), 255)
        self.draw = ImageDraw.Draw(self.image)
        self.last_x, self.last_y = None, None
        for char in self.alphabet:
            self.prob_labels[char].config(text="0.0%", foreground='black')
            self.prob_bars[char]['value'] = 0

    def process_and_predict(self):
        self.last_x, self.last_y = None, None
        inv_img = ImageOps.invert(self.image)
        bbox = inv_img.getbbox()
        if not bbox:
            return
        letter = inv_img.crop(bbox)
        w, h = letter.size
        max_dim = max(w, h)
        pad = int(max_dim * 0.3)
        container = Image.new('L', (max_dim + pad, max_dim + pad), 0)
        container.paste(letter, ((max_dim + pad - w)//2, (max_dim + pad - h)//2))
        letter_resized = container.resize((self.img_size, self.img_size), Image.Resampling.LANCZOS)
        img_array = np.array(letter_resized).astype(np.float32) / 255.0
        img_array = (img_array > 0.35).astype(np.float32)
        prev_show = letter_resized.resize((64, 64), Image.Resampling.NEAREST)
        self.ph_preview = ImageTk.PhotoImage(prev_show)
        self.preview_canvas.create_image(0, 0, anchor='nw', image=self.ph_preview)
        results = self.predictor.predict_all(img_array)
        top_char = results[0][0]
        for char, prob in results:
            percent = prob * 100
            self.prob_labels[char].config(text=f"{percent:>5.1f}%")
            self.prob_bars[char]['value'] = percent
            if char == top_char and prob > 0.1:
                self.prob_labels[char].config(foreground='green', font=('Courier', 12, 'bold'))
            else:
                self.prob_labels[char].config(foreground='black', font=('Courier', 12))

    def run(self):
        self.root.mainloop()

### Пояснения к приложению:

1. **KerasPredictor**: обёртка для предсказания Keras модели
2. **PyTorchPredictor**: обёртка для предсказания PyTorch модели
3. **DrawingApp**: GUI приложение с:
   - Холстом для рисования (400×400)
   - Отображением вероятностей для каждой буквы
   - Прогресс-барами и процентными значениями
   - Предпросмотром обработанного изображения

4. **Обработка изображения**:
   - Инверсия цветов
   - Кадрирование по контуру
   - Добавление полей (30% от размера)
   - Ресайз до 32×32
   - Бинаризация по порогу 0.35

In [ ]:
# Меню выбора модели для тестирования
def test_model(model_type):
    if model_type == 2:
        if not os.path.exists('model_keras.h5'):
            print("Ошибка: model_keras.h5 не найден!")
            return
        print("=== ЗАГРУЗКА KERAS МОДЕЛИ ===")
        def custom_tanh(x):
            return tf.nn.tanh(3.0 * x)
        keras.utils.get_custom_objects()['custom_tanh'] = custom_tanh
        model = keras.models.load_model('model_keras.h5')
        predictor = KerasPredictor(model, ALPHABET)
        title = "KERAS МОДЕЛЬ (tanh, α=3)"
    elif model_type == 3:
        if not os.path.exists('model_pytorch.pth'):
            print("Ошибка: model_pytorch.pth не найден!")
            return
        print("=== ЗАГРУЗКА PYTORCH МОДЕЛИ ===")
        pt_model = PyTorchNet(input_size, num_classes).to(device)
        pt_model.load_state_dict(torch.load('model_pytorch.pth', weights_only=True))
        predictor = PyTorchPredictor(pt_model, ALPHABET, device)
        title = "PYTORCH МОДЕЛЬ (tanh, α=3)"
    else:
        print("Неверный выбор!")
        return
    
    app = DrawingApp(predictor, ALPHABET, title)
    app.run()

# Запуск меню
print("\n" + "="*50)
print("ВЫБЕРИТЕ МОДЕЛЬ ДЛЯ ТЕСТИРОВАНИЯ:")
if os.path.exists('model_keras.h5'):
    print("  2 - Keras модель (tanh, α=3)")
if os.path.exists('model_pytorch.pth'):
    print("  3 - PyTorch модель (tanh, α=3)")
print("="*50)

available_choices = []
if os.path.exists('model_keras.h5'):
    available_choices.append('2')
if os.path.exists('model_pytorch.pth'):
    available_choices.append('3')

if not available_choices:
    print("Нет доступных моделей! Обучите модели.")
else:
    choice = input(f"Введите номер модели ({'/'.join(available_choices)}): ").strip()
    if choice in available_choices:
        test_model(int(choice))
    else:
        print("Неверный выбор!")

---
## 9. Сравнение реализаций

| Критерий | Собственная | Keras | PyTorch |
|----------|-------------|-------|----------|
| **Длина кода** | ~200 строк | ~50 строк | ~80 строк |
| **Гибкость** | Полная | Средняя | Высокая |
| **Скорость обучения** | Низкая | Высокая | Высокая |
| **GPU поддержка** | Нет | Да | Да |
| **Отладка** | Сложная | Средняя | Удобная |
| **Сохранение** | pickle | HDF5 | state_dict |

### Выводы:
1. **Keras** — самый лаконичный и удобный для быстрого прототипирования
2. **PyTorch** — оптимальный баланс между гибкостью и удобством
3. **Собственная реализация** — лучшее понимание внутренних процессов, но неэффективна для продакшена

### Влияние α=3 на tanh:
- Увеличивает градиенты в области насыщения
- Ускоряет сходимость
- Требует меньшего learning rate

---
## Заключение

В ходе лабораторной работы были реализованы и сравнены три подхода к созданию нейронной сети для классификации символов:

1. **Собственная реализация** на NumPy — дала глубокое понимание алгоритмов обратного распространения и SGD
2. **Keras** — показала высокую скорость разработки и удобство API
3. **PyTorch** — продемонстрировала гибкость и прозрачность вычислений

Все три реализации используют функцию активации **tanh(3x)**, которая обеспечивает более быстрое обучение по сравнению со стандартной tanh.

**Выполнил:** Привалихин Дмитрий Сергеевич, ИСИб-23-1